# EDA — HeyTea Dataset

Exploratory Data Analysis for slides Section 1.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image

ROOT     = Path('..') 
DATA_DIR = ROOT / 'tea_dataset'
df = pd.read_csv(DATA_DIR / 'metadata_clean.csv')

CLASSES   = ['cha_dam_yen','cha_khiao_nom','cha_nom_khai_muk','cha_phonlamai','cha_thai']
SHORT     = ['dam_yen','khiao_nom','nom_khai_muk','phonlamai','thai']
COLORS    = ['#e74c3c','#2ecc71','#9b59b6','#f39c12','#3498db']
print(f'Total images: {len(df)}')
print(df.groupby('class').size().to_string())

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
counts = [len(df[df['class']==c]) for c in CLASSES]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(SHORT, counts, color=COLORS, edgecolor='white')
axes[0].set_title('Class Distribution', fontsize=13)
axes[0].set_ylabel('Image count')
for i, v in enumerate(counts):
    axes[0].text(i, v + 2, str(v), ha='center', fontsize=10)

axes[1].pie(counts, labels=SHORT, colors=COLORS, autopct='%1.1f%%', startangle=140)
axes[1].set_title('Class Proportion', fontsize=13)

plt.suptitle('EDA — Class Distribution', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('eda_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Image size & aspect ratio ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['width'],  bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Width Distribution');  axes[0].set_xlabel('pixels')

axes[1].hist(df['height'], bins=30, color='coral',     edgecolor='white')
axes[1].set_title('Height Distribution'); axes[1].set_xlabel('pixels')

axes[2].hist(df['aspect_ratio'], bins=30, color='mediumseagreen', edgecolor='white')
axes[2].set_title('Aspect Ratio (W/H)');  axes[2].set_xlabel('ratio')

plt.suptitle('EDA — Image Size', fontsize=14)
plt.tight_layout()
plt.savefig('eda_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Width  : min={df['width'].min()}  max={df['width'].max()}  mean={df['width'].mean():.0f}")
print(f"Height : min={df['height'].min()} max={df['height'].max()} mean={df['height'].mean():.0f}")
print(f"Aspect : min={df['aspect_ratio'].min():.2f}  max={df['aspect_ratio'].max():.2f}")

In [ ]:
# ── Brightness distribution ───────────────────────────────────────────────────
# Compute mean pixel brightness for each image (skip if slow)
brightness = []
for _, row in df.iterrows():
    try:
        img = Image.open(DATA_DIR / row['relpath']).convert('L')
        brightness.append(np.array(img).mean())
    except:
        brightness.append(np.nan)
df['brightness'] = brightness

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['brightness'].dropna(), bins=40, color='gold', edgecolor='white')
ax.set_title('Brightness Distribution (mean pixel intensity)')
ax.set_xlabel('Mean pixel value (0–255)')
ax.axvline(df['brightness'].mean(), color='red', linestyle='--', label=f"mean={df['brightness'].mean():.1f}")
ax.legend()
plt.tight_layout()
plt.savefig('eda_brightness.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Color vs Grayscale ────────────────────────────────────────────────────────
gray_count  = df['is_grayscale'].sum()
color_count = len(df) - gray_count
print(f'Color images:     {color_count} ({color_count/len(df)*100:.1f}%)')
print(f'Grayscale images: {gray_count}  ({gray_count/len(df)*100:.1f}%)')

# Per-class grayscale breakdown
print('\nGrayscale per class:')
print(df.groupby('class')['is_grayscale'].sum().to_string())

In [ ]:
# ── Sample images grid ────────────────────────────────────────────────────────
N_PER_CLASS = 4
fig, axes = plt.subplots(len(CLASSES), N_PER_CLASS,
                          figsize=(N_PER_CLASS * 2.5, len(CLASSES) * 2.5))

for r, cls in enumerate(CLASSES):
    samples = df[df['class'] == cls].sample(min(N_PER_CLASS, len(df[df['class']==cls])),
                                             random_state=42)
    for c, (_, row) in enumerate(samples.iterrows()):
        try:
            img = Image.open(DATA_DIR / row['relpath']).convert('RGB')
            img.thumbnail((224, 224))
            axes[r][c].imshow(img)
        except:
            axes[r][c].text(0.5, 0.5, 'N/A', ha='center', va='center')
        axes[r][c].axis('off')
        if c == 0:
            axes[r][c].set_ylabel(cls.replace('cha_',''), fontsize=9, rotation=90, labelpad=40)

plt.suptitle('Sample Images per Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('eda_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary stats table (for slides) ─────────────────────────────────────────
summary = df.groupby('class').agg(
    count=('filename','count'),
    avg_width=('width','mean'),
    avg_height=('height','mean'),
    grayscale=('is_grayscale','sum'),
).round(0).astype(int)
summary['color'] = summary['count'] - summary['grayscale']
print(summary.to_string())